<a href="https://colab.research.google.com/github/apHub27/sveltetech-ai-internship/blob/main/rnn_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ========================================================
# STEP 1: TEXT DATA SIMULATION (SAME AS LSTM)
# ========================================================
print("📥 Preparing text data vectors for RNN...")
dummy_text_vectors = torch.randint(low=1, high=1000, size=(500, 20)) # 500 reviews, 20 words each
dummy_labels = torch.randint(low=0, high=2, size=(500,)) # 1 = Positive, 0 = Negative

dataset = TensorDataset(dummy_text_vectors, dummy_labels)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)


# ========================================================
# STEP 2: PLAIN RNN ARCHITECTURE DESIGN
# ========================================================
class MovieSentimentRNN(nn.Module):
    def __init__(self):
        super(MovieSentimentRNN, self).__init__()

        # Word ko vector me badalne ke liye
        self.embedding = nn.Embedding(num_embeddings=1000, embedding_dim=64)

        # 🔥 CHANGE HERE: Humne plain nn.RNN use kiya hai (Isme simple loop hota hai, gates nahi hote)
        self.rnn = nn.RNN(input_size=64, hidden_size=128, batch_first=True)

        # Final classification layer
        self.fc = nn.Linear(128, 2)

    def forward(self, x):
        x = self.embedding(x)

        rnn_out, hidden_state = self.rnn(x)

        # Poora sentence padhne ke baad aakhri word ki memory uthana
        final_memory = rnn_out[:, -1, :]

        out = self.fc(final_memory)
        return out

model = MovieSentimentRNN()


# ========================================================
# STEP 3: LOSS & OPTIMIZER
# ========================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)


# ========================================================
# STEP 4: THE TRAINING LOOP
# ========================================================
print("\n🚀 Training the Plain RNN Text Model...")
epochs = 3

for epoch in range(epochs):
    running_loss = 0.0
    for batch_text, batch_labels in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_text)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"🏅 Epoch [{epoch+1}/{epochs}] | RNN Text Loss: {running_loss/len(train_loader):.4f}")

# Model save karna
torch.save(model.state_dict(), 'rnn_sentiment_model.pth')
print("\n💾 Success! Model saved as 'rnn_sentiment_model.pth'")


📥 Preparing text data vectors for RNN...

🚀 Training the Plain RNN Text Model...
🏅 Epoch [1/3] | RNN Text Loss: 0.7671
🏅 Epoch [2/3] | RNN Text Loss: 0.5966
🏅 Epoch [3/3] | RNN Text Loss: 0.4530

💾 Success! Model saved as 'rnn_sentiment_model.pth'


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ========================================================
# STEP 1: TEXT DATA SIMULATION (IMDB FORMAT)
# ========================================================
print("📥 Preparing text data vectors (Movie Reviews)...")

# Maano hamare paas 500 reviews hain, aur har review me exactly 20 words hain.
# Har word ko humne ek number (Token ID) de diya hai.
dummy_text_vectors = torch.randint(low=1, high=1000, size=(500, 20)) # 500 reviews, 20 words each
dummy_labels = torch.randint(low=0, high=2, size=(500,)) # 1 = Positive Review, 0 = Negative Review

# Loader me wrap karna (Batch Size = 32)
dataset = TensorDataset(dummy_text_vectors, dummy_labels)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
print("✅ Text Dataset is ready!")


# ========================================================
# STEP 2: LSTM ARCHITECTURE DESIGN
# ========================================================
class MovieSentimentLSTM(nn.Module):
    def __init__(self):
        super(MovieSentimentLSTM, self).__init__()

        # 1. Embedding Layer: Yeh numbers (words) ko meaningful geometric vectors me badalta hai
        self.embedding = nn.Embedding(num_embeddings=1000, embedding_dim=64)

        # 2. LSTM Layer: Yeh hamari looping memory hai jo word-by-word context yaad rakhti hai
        self.lstm = nn.LSTM(input_size=64, hidden_size=128, batch_first=True)

        # 3. Output Layer: Final decision (Positive vs Negative) ke liye 2 outputs
        self.fc = nn.Linear(128, 2)

    def forward(self, x):
        # x shape: [batch_size, sequence_length] -> [32, 20]
        x = self.embedding(x)  # Now shape becomes [32, 20, 64]

        # LSTM output deta hai aur sath me hidden state (memory) bhi return karta hai
        lstm_out, (hidden_state, cell_state) = self.lstm(x)

        # Hum sirf sabse aakhri word padhne ke baad waali final memory status uthate hain
        final_memory = lstm_out[:, -1, :] # Shape: [32, 128]

        out = self.fc(final_memory)
        return out

# Model instance banana
model = MovieSentimentLSTM()


# ========================================================
# STEP 3: LOSS & OPTIMIZER
# ========================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)


# ========================================================
# STEP 4: THE TRAINING LOOP
# ========================================================
print("\n🚀 Training the LSTM Text Model...")
epochs = 3

for epoch in range(epochs):
    running_loss = 0.0
    for batch_text, batch_labels in train_loader:
        optimizer.zero_grad()

        outputs = model(batch_text)
        loss = criterion(outputs, batch_labels)

        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"🏅 Epoch [{epoch+1}/{epochs}] | Text Loss: {running_loss/len(train_loader):.4f}")

# Model save karna
torch.save(model.state_dict(), 'lstm_sentiment_model.pth')
print("\n💾 Success! Model saved as 'lstm_sentiment_model.pth'")


📥 Preparing text data vectors (Movie Reviews)...
✅ Text Dataset is ready!

🚀 Training the LSTM Text Model...
🏅 Epoch [1/3] | Text Loss: 0.6981
🏅 Epoch [2/3] | Text Loss: 0.5433
🏅 Epoch [3/3] | Text Loss: 0.2738

💾 Success! Model saved as 'lstm_sentiment_model.pth'
